In [ ]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Resolve absolute paths to local Silver Delta table directories
PROJECT_ROOT = os.path.dirname(os.getcwd())

TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_kafka_stream")
TARGET_DLQ_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_kafka_stream_dlq")

# Initialize SparkSession with Delta support
builder = (
    SparkSession.builder.appName("Query_Silver")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
silver_df = spark.read.format("delta").load(TARGET_PATH)
print(f"Total Clean Network Telemetry Records: {silver_df.count():,}")
silver_df.select("host", "src_ip", "dst_ip", "protocol", "src_port", "dst_port", "_processed_at").show(10, truncate=False)

In [ ]:
# Load Silver DLQ Delta table
dlq_df = spark.read.format("delta").load(TARGET_DLQ_PATH)

print(f"Total Quarantined DLQ Records: {dlq_df.count()}")
dlq_df.select("event_time", "event_type", "price", "_dlq_reason", "_source_file").show(10, truncate=False)

In [ ]:
# KAFKA STREAMS
dlq_df = spark.read.format("delta").load(TARGET_DLQ_PATH)

print(f"Total Quarantined Network Logs (DLQ): {dlq_df.count():,}")
dlq_df.select("host", "message", "_dlq_reason", "_processed_at").show(10, truncate=False)
